# 🌬️ Kelmarsh Wind Farm — SCADA Data Exploration

This notebook walks through:
1. Loading the Kelmarsh dataset via `KelmarshConnector`
2. Signal inspection and data quality analysis
3. Power curve computation (IEC 61400-12-1)
4. Anomaly detection with IsolationForest
5. KPI summary

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.dates as mdates
from pathlib import Path

plt.style.use('dark_background')
plt.rcParams.update({'figure.facecolor': '#0a0a0a', 'axes.facecolor': '#111111'})

print('Imports OK')

In [ ]:
# ── Load sample data ──
from connectors.kelmarsh.loader import KelmarshConnector

SAMPLE_PATH = Path('../data/sample/kelmarsh_K1_sample.csv')

conn = KelmarshConnector(SAMPLE_PATH)
df = conn.load_dataframe('K1')

print(f'Loaded {len(df):,} rows')
print(f'Date range: {df.index.min()} → {df.index.max()}')
df.head()

In [ ]:
# ── Data quality overview ──
quality = pd.DataFrame({
    'available': df.notna().sum(),
    'missing': df.isna().sum(),
    'completeness_%': (df.notna().mean() * 100).round(1),
    'mean': df.mean(numeric_only=True).round(2),
    'std': df.std(numeric_only=True).round(2),
})
quality

In [ ]:
# ── Wind speed & power time series ──
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(14, 6), sharex=True)

ax1.plot(df.index, df['wind_speed_ms'], color='#22d3ee', lw=0.8, alpha=0.8)
ax1.set_ylabel('Wind Speed (m/s)', color='#22d3ee')
ax1.set_ylim(0)

ax2.fill_between(df.index, df['active_power_kw'].clip(lower=0), alpha=0.7, color='#34d399')
ax2.axhline(2050, color='#ef4444', lw=0.8, linestyle='--', label='Rated 2050 kW')
ax2.set_ylabel('Active Power (kW)', color='#34d399')
ax2.set_ylim(0, 2200)
ax2.legend(fontsize=8)

fig.suptitle('Kelmarsh K1 — Wind & Power (Sample)', fontsize=12, color='white')
plt.tight_layout()
plt.show()

In [ ]:
# ── Power curve scatter ──
fig, ax = plt.subplots(figsize=(9, 5))

valid = df[df['wind_speed_ms'].notna() & df['active_power_kw'].notna()]
sc = ax.scatter(
    valid['wind_speed_ms'],
    valid['active_power_kw'].clip(lower=0),
    c=valid['active_power_kw'].clip(lower=0),
    cmap='plasma',
    s=4,
    alpha=0.5
)
plt.colorbar(sc, ax=ax, label='Power (kW)')
ax.axhline(2050, color='#ef4444', lw=1, linestyle='--', label='Rated power')
ax.set_xlabel('Wind Speed (m/s)')
ax.set_ylabel('Active Power (kW)')
ax.set_title('Power Curve Scatter — Kelmarsh K1')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# ── IEC Power curve ──
from analytics.power_curve.iec_power_curve import compute_power_curve

# For demo purposes, we use a larger synthetic dataset
from tests.test_power_curve import make_synthetic_scada
big_df = make_synthetic_scada(n=10000)

curve = compute_power_curve(big_df, turbine_id='K1_SIM')
print(f'Cut-in: {curve.cut_in_ms:.1f} m/s')
print(f'Cut-out: {curve.cut_out_ms:.1f} m/s')
print(f'Capacity factor: {curve.capacity_factor:.2%}')
print(f'AEP estimate: {curve.annual_energy_production_mwh:.0f} MWh/year')

In [ ]:
# ── Anomaly detection ──
from analytics.anomaly.detectors import IsolationForestDetector

detector = IsolationForestDetector(contamination=0.03)
detector.fit(big_df)

scores = detector.score(big_df)
anomalies = scores[scores > 0.7]
print(f'Anomalies detected: {len(anomalies)} ({len(anomalies)/len(big_df)*100:.1f}%)')

fig, ax = plt.subplots(figsize=(12, 3))
ax.plot(scores.index, scores.values, lw=0.5, color='#6366f1', alpha=0.7)
ax.axhline(0.7, color='#ef4444', lw=1, linestyle='--', label='Threshold 0.7')
ax.fill_between(scores.index, scores.values, 0.7,
                where=scores.values > 0.7, alpha=0.4, color='#ef4444')
ax.set_ylabel('Anomaly Score')
ax.set_title('IsolationForest Anomaly Scores')
ax.legend()
plt.tight_layout()
plt.show()